# Thesis — 150-sentence journal profile (10 training seeds)

1. **Runtime:** Google Colab with **GPU** (recommended).
2. **Repo:** public — no GitHub token.
3. **Profile:** `--journal-paper` (Exp04 λ grid, Exp01/04 + OOF SVM/RF fusion, 5-fold CV).
4. **Seeds:** `--num-seeds 10` (overrides journal default of 3).
5. **Data:** uses git-tracked **`data/ner_dataset_150_seed42.pkl`** (fixed 150 sentences; full corpus: `ner_dataset_full.pkl`).
6. **Outputs:** everything writes to **`5folds_run/`** on Drive (separate from older cross-comparison folders).

Docs: `thesis_overview.md` Part IV, `COLAB_README.md`.

## 1 — Mount Google Drive (recommended)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 — Paths (edit `WORK_ROOT` if needed)

In [ ]:
import os
from pathlib import Path

WORK_ROOT = Path("/content/drive/MyDrive/thesis_project")
REPO_DIR = WORK_ROOT / "thesis"
REPO_URL = "https://github.com/menachemsperka/thesis.git"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_ROOT)
print("WORK_ROOT:", WORK_ROOT)
print("REPO_DIR:", REPO_DIR)

## 3 — Clone or update repository (public, no token)

In [ ]:
import os
import subprocess

if REPO_DIR.is_dir() and (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    print("Updated existing clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    print("Cloned repository.")

os.chdir(REPO_DIR)
!git log -1 --oneline

## 4 — Install dependencies

In [ ]:
import os
os.chdir(REPO_DIR)
!pip install -q -r requirements.txt

## 5 — Colab environment (set before running experiments)

In [ ]:
import os

os.environ["THESIS_RUN_ENV"] = "colab"
os.environ["WANDB_DISABLED"] = "true"
os.environ.setdefault("THESIS_DELETE_MODELS_AFTER_TRAIN", "1")

print("THESIS_RUN_ENV =", os.environ["THESIS_RUN_ENV"])

## 6 — Check labeled dataset under `data/`

In [ ]:
from pathlib import Path

data_dir = REPO_DIR / "data"
NER_PKL_150 = data_dir / "ner_dataset_150_seed42.pkl"
NER_PKL_FULL = data_dir / "ner_dataset_full.pkl"
for label, p in (
    ("150-sent (this notebook)", NER_PKL_150),
    ("full ~300-sent", NER_PKL_FULL),
):
    print(f"{label}: {p.name} -> {'OK' if p.is_file() else 'MISSING'}")

if not NER_PKL_150.is_file():
    raise FileNotFoundError(
        "Missing ner_dataset_150_seed42.pkl — git pull main or run "
        "python scripts/build_ner_dataset_corpus_pkls.py"
    )

## 7 — Run configuration

In [ ]:
from pathlib import Path

# Isolated run root — all artifacts, splits, seeds, and checkpoints live here only.
OUTPUT_DIR = WORK_ROOT / "5folds_run"
NUM_SEEDS = 10
MODELS = "dictabert,berel"
NER_DATASET_PKL = REPO_DIR / "data" / "ner_dataset_150_seed42.pkl"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("NUM_SEEDS:", NUM_SEEDS)

## 8 — Run cross-comparison (argv one row per argument)

Uses **`--journal-paper`** with **`--num-seeds 10`**. Same command split into a Python list so each flag is its own line.

Expect a long run; use **`--resume`** on reconnect (re-run cells 1–7, then this cell only).

In [ ]:
import os
import sys
import subprocess

os.chdir(REPO_DIR)
os.environ["THESIS_RUN_ENV"] = "colab"
os.environ["WANDB_DISABLED"] = "true"
os.environ["THESIS_NER_CSV"] = str(NER_DATASET_PKL)

argv = [
    sys.executable,
    "run_cross_data_model_comparison.py",
    "--journal-paper",
    "--num-seeds",
    str(NUM_SEEDS),
    "--resume",
    "--output-dir",
    str(OUTPUT_DIR),
    "--exp07-source",
    "rerun",
    "--models",
    MODELS,
    "--base-mode",
    "auto",
]

print("Command:")
for i, part in enumerate(argv):
    print(f"  [{i}] {part}")

subprocess.run(argv, check=True)

## 9 — Optional: rebuild Excel/JSON only (no retraining)

In [ ]:
import os
import sys
import subprocess

os.chdir(REPO_DIR)

argv_rebuild = [
    sys.executable,
    "run_cross_data_model_comparison.py",
    "--rebuild-from-checkpoint",
    "--output-dir",
    str(OUTPUT_DIR),
]

subprocess.run(argv_rebuild, check=True)

## 10 — List key output files

In [ ]:
from pathlib import Path

out = Path(OUTPUT_DIR)
checks = [
    "run_manifest.json",
    "training_seeds.json",
    "exp04_lambda_grid_cache/exp04_lambda_selection.json",
    "exp07/splits/split_meta.json",
    "cross_comparison_progress_latest.json",
    "cross_comparison_latest.xlsx",
]
for rel in checks:
    p = out / rel
    print(f"{rel}: {'OK' if p.is_file() else 'missing'}")

for p in sorted(out.glob("consolidated_error_analysis*.xlsx"))[-2:]:
    print("workbook:", p.name)